In [10]:
import os
import sys
import torch
import gc
import time
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- 1. Environment & Pathing ---
root = Path("/home/jupyter-1nt23cb058/Capstone")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# --- 2. SpatialPartition Class Declaration ---
# Must match Phase 11.1 definition exactly for unpickling
class SpatialPartition:
    def __init__(self, cid, node_ids, edge_index, edge_attr, train_mask):
        self.cid = int(cid)
        self.n_id = torch.as_tensor(node_ids, dtype=torch.long)
        self.edge_index = edge_index.long()
        self.edge_attr = edge_attr.float()
        self.train_mask = train_mask.bool()
        self.val_mask = None
        self.upwind_edge_mask = torch.zeros(edge_index.shape[1], dtype=torch.bool)
        self.x = None

# --- 3. Imports & Module Reload ---
import importlib
import gnn.model
importlib.reload(gnn.model)
from gnn.model import STPIGNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 4. Loading Cached Data & Verified Weights ---
cluster_data_path = root / "data/processed/graph/partitioned_manifold_64_clusters.pt"
best_weight_path = root / "notebooks/models/phase_11_1_final/stpignn_mesh_resolution_E11_20260429_1205.pt"

cluster_data = torch.load(cluster_data_path, weights_only=False, map_location=device)
checkpoint = torch.load(best_weight_path, weights_only=False, map_location=device)

# --- 5. Model Initialization with Alignment Patch ---
# We start with node_in_dim=16 to match your 16-channel feature manifold
model = STPIGNN(
    node_in_dim=16,
    edge_dim=1,
    spatial_hidden_dim=96,
    temporal_hidden_dim=96,
    gnn_layers=2
).to(device)

# --- 6. The State Dict Alignment Logic ---
state_dict = checkpoint['state_dict']

# Detecting the mismatch mentioned in your error log
# Checkpoint has [96, 16], Model currently has [96, 1]
ck_gnn_shape = state_dict['gnn_layers.0.lin.weight'].shape
md_gnn_shape = model.gnn_layers[0].lin.weight.shape

print(f"🔍 Audit: Checkpoint GNN Shape: {ck_gnn_shape}")
print(f"🔍 Audit: Current Model GNN Shape: {md_gnn_shape}")

if ck_gnn_shape != md_gnn_shape:
    print("\n⚠️ Shape mismatch detected in GNN layers. Applying Architecture Patch...")
    import torch.nn as nn
    for i in range(len(model.gnn_layers)):
        # Re-initialize the internal linear layers of the GNN to match the checkpoint
        # This ensures the 16 features from Phase 11.1 map correctly to the 96 hidden dims
        model.gnn_layers[i].lin = nn.Linear(16, 96).to(device)
    print("✅ Architecture Patched.")

# --- 7. Final Load & Verification ---
try:
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    print("\n🚀 FULL MATCH SUCCESSFUL: Phase 11.1 Weights Injected.")
except RuntimeError as e:
    print(f"\n❌ Load Failed: {e}")

# --- 8. Manifold Integrity Audit ---
total_energy = sum(c.x.sum().item() for c in cluster_data if c.x is not None)
print("-" * 30)
print(f"📊 Manifold Clusters: {len(cluster_data)}")
print(f"⚡ Manifold Energy:   {total_energy:.2f} (Target: ~1898.62)")
print("-" * 30)

🔍 Audit: Checkpoint GNN Shape: torch.Size([96, 16])
🔍 Audit: Current Model GNN Shape: torch.Size([96, 1])

⚠️ Shape mismatch detected in GNN layers. Applying Architecture Patch...
✅ Architecture Patched.

🚀 FULL MATCH SUCCESSFUL: Phase 11.1 Weights Injected.
------------------------------
📊 Manifold Clusters: 64
⚡ Manifold Energy:   1898.62 (Target: ~1898.62)
------------------------------
